# Video Normalization Pipeline
Normalizes video FPS to 60 and upscales resolution 4x using EDSR super-resolution.

**Run cells top to bottom.**

In [ ]:
# Cell 1 — Install dependencies
!pip install -q opencv-contrib-python
!apt-get install -y ffmpeg

In [ ]:
# Cell 2 — Download EDSR x4 model (~37 MB)
!wget -q --show-progress -O EDSR_x4.pb \
  "https://github.com/Saafke/EDSR_Tensorflow/raw/master/models/EDSR_x4.pb"
print("Model ready:", __import__('os').path.getsize('EDSR_x4.pb') // 1_000_000, "MB")

In [ ]:
# Cell 3 — Upload pipeline.py
from google.colab import files
print("Upload pipeline.py from your computer:")
files.upload()

In [ ]:
# Cell 4 — Upload your .mp4 videos
from google.colab import files
import os

os.makedirs("videos", exist_ok=True)
print("Upload your .mp4 files:")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"videos/{name}", "wb") as f:
        f.write(data)
print("Uploaded to videos/:", list(uploaded.keys()))

In [ ]:
# Cell 5 — Run the pipeline
import os, time, shutil, tempfile, sys
sys.path.insert(0, ".")
from pipeline import fps_normalize, upscale_video

MODEL      = "EDSR_x4.pb"
VIDEOS_DIR = "videos"
OUTPUT_DIR = "final_videos"
TARGET_FPS = 60
SCALE      = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
mp4_files = [f for f in os.listdir(VIDEOS_DIR) if f.endswith(".mp4")]
print(f"Found {len(mp4_files)} video(s): {mp4_files}")

tmp_dir = tempfile.mkdtemp()
results = []
try:
    for filename in mp4_files:
        src       = os.path.join(VIDEOS_DIR, filename)
        tmp_dst   = os.path.join(tmp_dir, filename)
        final_dst = os.path.join(OUTPUT_DIR, filename)
        print(f"\n[{filename}] Starting...")
        try:
            t0 = time.time()
            fps_normalize(src, tmp_dst, TARGET_FPS)
            t1 = time.time()
            upscale_video(tmp_dst, final_dst, MODEL, SCALE)
            t2 = time.time()
            print(f"[{filename}] Done in {t2-t0:.1f}s -> {final_dst}")
            results.append((filename, True))
        except Exception as e:
            print(f"[{filename}] FAILED: {e}")
            results.append((filename, False))
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

succeeded = sum(1 for _, ok in results if ok)
print(f"\nComplete: {succeeded}/{len(results)} succeeded")

In [ ]:
# Cell 6 — Download results to your computer
from google.colab import files
import os

output_files = os.listdir("final_videos")
print(f"Downloading {len(output_files)} file(s)...")
for f in output_files:
    files.download(f"final_videos/{f}")